In [ ]:
# %% [1] Setup
from google.colab import drive
drive.mount('/content/drive')
!pip install monai

import os, glob, json, time, random
import numpy as np, pandas as pd, nibabel as nib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import wilcoxon, pearsonr
from monai.networks.nets import SwinUNETR
from monai.losses import DiceFocalLoss
from monai.metrics import compute_hausdorff_distance
from monai.transforms import RandAffined, KeepLargestConnectedComponent
from monai.data import DataLoader as MonaiDataLoader

device       = torch.device("cuda")
PROJECT      = "/content/drive/My Drive/Research Project"
ROOT_DIR     = f"{PROJECT}/Data"
OUT_DIR      = f"{PROJECT}/Models_v6"
SPLIT_FILE   = f"{PROJECT}/Models_v2/split.json"
TARGET_SHAPE = (128, 128, 96)
SEED         = 42
os.makedirs(OUT_DIR, exist_ok=True)
assert OUT_DIR.endswith("Models_v6"), OUT_DIR

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()

In [ ]:
# %% [2] Loading the subject split
split = json.load(open(SPLIT_FILE))
subj = {k: {p["subject"] for p in v} for k, v in split.items()}
assert not (subj["train"] & subj["val"] or subj["train"] & subj["test"] or subj["val"] & subj["test"]), \
    "Subject overlap between splits"
for k, v in split.items():
    print(k, len(v), {c: sum(p["cat"] == c for p in v) for c in ["AD", "CN", "MCI"]})

In [ ]:
# %% [3] Region of interest from training labels
coords = {}
for k in split:
    for p in split[k]:
        coords[p["seg"]] = np.argwhere(nib.load(p["seg"]).get_fdata() >= 1)

RAW_SHAPE = np.array(nib.load(split["train"][0]["brain"]).shape)
T = np.array(TARGET_SHAPE)
lo = np.min([coords[p["seg"]].min(0) for p in split["train"]], axis=0)
hi = np.max([coords[p["seg"]].max(0) for p in split["train"]], axis=0)
assert np.all(hi - lo + 1 <= T), f"hippocampus extent {hi - lo + 1} exceeds ROI {T}"
ROI_START = np.clip((lo + hi) // 2 - T // 2, 0, RAW_SHAPE - T).astype(int)
json.dump(ROI_START.tolist(), open(os.path.join(OUT_DIR, "roi_start.json"), "w"))
print("raw shape:", RAW_SHAPE, "| train extent:", hi - lo + 1, "| ROI start:", ROI_START)

for k in split:
    lost = [p["subject"] for p in split[k]
            if not np.all((coords[p["seg"]] >= ROI_START) & (coords[p["seg"]] < ROI_START + T))]
    print(k, "| truncated:", lost)
    assert not lost, f"{k}: hippocampus outside ROI for {lost}"
print("All subjects fully inside the ROI.")

In [ ]:
# %% [4] Dataset and loaders
class HarPDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs, self.augment = pairs, augment
        self.affine = RandAffined(keys=["img", "seg"], prob=0.5,
                                  rotate_range=(0.17, 0.17, 0.17),     # ±10 degrees
                                  scale_range=(0.1, 0.1, 0.1),         # ±10 %
                                  mode=("bilinear", "nearest"), padding_mode="zeros")
    def __len__(self): return len(self.pairs)

    @staticmethod
    def _load(path):
        return torch.tensor(nib.load(path).get_fdata(), dtype=torch.float32)

    @staticmethod
    def _crop(x):
        s, t = ROI_START, TARGET_SHAPE
        return x[s[0]:s[0] + t[0], s[1]:s[1] + t[1], s[2]:s[2] + t[2]]

    def __getitem__(self, i):
        p = self.pairs[i]
        img  = self._crop(self._load(p["brain"])).clone()
        mask = img > 0
        img[mask] = (img[mask] - img[mask].mean()) / (img[mask].std() + 1e-8)
        img = img.unsqueeze(0)                                          # (1,H,W,D)
        lbl = self._crop(self._load(p["seg"])).long().unsqueeze(0).float()   # (1,H,W,D), 0/1

        # Canonical side: mirror so the annotated hippocampus always lies in the low-x half
        if lbl[0].nonzero()[:, 0].float().mean() >= TARGET_SHAPE[0] / 2:
            img, lbl = img.flip(1), lbl.flip(1)

        if self.augment:                                                # no left-right flip
            d = self.affine({"img": img, "seg": lbl})
            img, lbl = torch.as_tensor(d["img"]), torch.as_tensor(d["seg"])
            a = 0.9 + 0.2 * torch.rand(1).item()
            b = -0.1 + 0.2 * torch.rand(1).item()
            img = torch.where(img != 0, img * a + b, img)

        seg = F.one_hot(lbl[0].round().long(), 2).permute(3, 0, 1, 2).float()   # (2,H,W,D)
        return img.float(), seg, p["subject"]

g = torch.Generator().manual_seed(SEED)
train_loader = MonaiDataLoader(HarPDataset(split["train"], augment=True), batch_size=1,
                               shuffle=True, generator=g, num_workers=2)
val_loader   = DataLoader(HarPDataset(split["val"]),  batch_size=1, shuffle=False, num_workers=2)
test_loader  = DataLoader(HarPDataset(split["test"]), batch_size=1, shuffle=False, num_workers=2)

x, y, _ = next(iter(train_loader))
print("input:", tuple(x.shape), "| label:", tuple(y.shape), "| hippocampus voxels:", int(y[0, 1].sum()))

In [ ]:
# %% [5] Check: annotated side per split, and every item ends up on the canonical side
for k in split:
    xs = [coords[p["seg"]][:, 0].mean() for p in split[k]]
    print(k, "| annotated low-x:", sum(x < RAW_SHAPE[0] / 2 for x in xs),
          "| annotated high-x:", sum(x >= RAW_SHAPE[0] / 2 for x in xs))
for k, loader in [("val", val_loader), ("test", test_loader)]:
    off = [sid[0] for _, seg, sid in loader
           if seg[0, 1].nonzero()[:, 0].float().mean() >= TARGET_SHAPE[0] / 2]
    print(k, "| items NOT on canonical side after loading:", off)
    assert not off

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
for a, i in zip(ax, range(4)):
    img, seg, sid = HarPDataset(split["train"])[i]
    z = int(seg[1].sum(dim=(0, 1)).argmax())
    a.imshow(img[0, :, :, z].T, cmap="gray", origin="lower"); a.contour(seg[1, :, :, z].T, colors="r")
    a.set_title(sid)
plt.suptitle("Annotated hippocampus should be on the same side in all four"); plt.show()

In [ ]:
# %% [6] Models
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p=0.0):
        super().__init__()
        layers = [
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True), nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True), nn.LeakyReLU(0.01, inplace=True),
        ]
        if dropout_p > 0: layers.append(nn.Dropout3d(p=dropout_p))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, base_filters=16):
        super().__init__()
        f = base_filters
        self.enc1 = DoubleConv(in_channels, f); self.pool1 = nn.MaxPool3d(2)
        self.enc2 = DoubleConv(f, f*2);         self.pool2 = nn.MaxPool3d(2)
        self.enc3 = DoubleConv(f*2, f*4);       self.pool3 = nn.MaxPool3d(2)
        self.enc4 = DoubleConv(f*4, f*8);       self.pool4 = nn.MaxPool3d(2)
        self.bottleneck = DoubleConv(f*8, f*16, dropout_p=0.3)
        self.up4 = nn.ConvTranspose3d(f*16, f*8, 2, stride=2); self.dec4 = DoubleConv(f*16, f*8, dropout_p=0.2)
        self.up3 = nn.ConvTranspose3d(f*8,  f*4, 2, stride=2); self.dec3 = DoubleConv(f*8,  f*4, dropout_p=0.1)
        self.up2 = nn.ConvTranspose3d(f*4,  f*2, 2, stride=2); self.dec2 = DoubleConv(f*4,  f*2)
        self.up1 = nn.ConvTranspose3d(f*2,  f,   2, stride=2); self.dec1 = DoubleConv(f*2,  f)
        self.out_conv = nn.Conv3d(f, out_channels, 1)
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.InstanceNorm3d) and m.affine:
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    @staticmethod
    def _pad(up, skip):
        diff = [s - u for s, u in zip(skip.shape[2:], up.shape[2:])]
        pad  = [d for d in reversed(diff) for _ in (0, 1)]
        return F.pad(up, pad) if any(p != 0 for p in pad) else up

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool1(e1)); e3 = self.enc3(self.pool2(e2)); e4 = self.enc4(self.pool3(e3))
        b  = self.bottleneck(self.pool4(e4))
        d4 = self.dec4(torch.cat([self._pad(self.up4(b),  e4), e4], 1))
        d3 = self.dec3(torch.cat([self._pad(self.up3(d4), e3), e3], 1))
        d2 = self.dec2(torch.cat([self._pad(self.up2(d3), e2), e2], 1))
        d1 = self.dec1(torch.cat([self._pad(self.up1(d2), e1), e1], 1))
        return self.out_conv(d1)

def build_unet():
    m = UNet3D().to(device)
    with torch.no_grad():
        m.out_conv.bias.copy_(torch.tensor([2.0, -2.0], device=device))
    return m

def build_teacher():
    return SwinUNETR(in_channels=1, out_channels=2, feature_size=24,
                     spatial_dims=3, use_checkpoint=True).to(device)

n_params = lambda m: sum(p.numel() for p in m.parameters())

In [ ]:
# %% [7] Loss, validation metric, training loop
sup_loss = DiceFocalLoss(softmax=True, gamma=2.0, alpha=0.95)

@torch.no_grad()
def fg_dice(logits, seg):
    pred, gt = logits.argmax(1) == 1, seg[:, 1] > 0.5
    tp, denom = (pred & gt).sum().item(), pred.sum().item() + gt.sum().item()
    return 2 * tp / denom if denom else 1.0

def train(model, name, epochs, lr, wd, teacher=None, lam_sup=0.90, lam_kd=0.10,
          patience=None, use_amp=True, clip=1.0):
    """Best epoch chosen on val Dice. Re-running the cell resumes from the last epoch."""
    ckpt_best = os.path.join(OUT_DIR, f"{name}_best.pth")
    ckpt_last = os.path.join(OUT_DIR, f"{name}_last.pth")
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    start, best, bad, hist = 0, -1.0, 0, []

    if os.path.exists(ckpt_last):
        s = torch.load(ckpt_last, map_location=device)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"])
        sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
        start, best, bad, hist = s["epoch"] + 1, s["best"], s["bad"], s["hist"]
        print(f"Resuming {name} from epoch {start + 1}")

    for ep in range(start, epochs):
        model.train(); tl, t0 = 0.0, time.time()
        for img, seg, _ in train_loader:
            img, seg = img.to(device), seg.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=use_amp):
                logits = model(img)
                loss = lam_sup * sup_loss(logits, seg)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(img)
            if teacher is not None:                                  # KD term in full precision
                p_s = torch.softmax(logits.float(), 1)[:, 1]
                p_t = torch.softmax(t_logits.float(), 1)[:, 1]
                loss = loss + lam_kd * F.mse_loss(p_s, p_t)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
            tl += loss.item()
        sched.step()

        model.eval()
        with torch.no_grad():
            vd = float(np.mean([fg_dice(model(i.to(device)), s.to(device)) for i, s, _ in val_loader]))
        hist.append(dict(epoch=ep + 1, train_loss=tl / len(train_loader), val_dice=vd))
        print(f"{name} | epoch {ep + 1}/{epochs} | loss {tl / len(train_loader):.4f} | "
              f"val Dice {vd:.4f} | best {max(best, vd):.4f} | {time.time() - t0:.0f}s")

        if vd > best:
            best, bad = vd, 0
            torch.save({"state_dict": model.state_dict(), "epoch": ep + 1, "val_dice": vd}, ckpt_best)
        else:
            bad += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep, best=best, bad=bad, hist=hist), ckpt_last)
        if patience and bad >= patience:
            print(f"Early stopping {name} at epoch {ep + 1}"); break

    pd.DataFrame(hist).to_csv(os.path.join(OUT_DIR, f"{name}_history.csv"), index=False)
    print(f"{name}: best val Dice {best:.4f}")
    return best

In [ ]:
# %% [8] Train teacher
seed_everything()
teacher = build_teacher()
print("teacher params:", n_params(teacher))                         # expected 15,703,004
train(teacher, "teacher", epochs=100, lr=1e-4, wd=1e-5, lam_sup=1.0, clip=None)


In [ ]:
# %% [9] Freezing best teacher
teacher = build_teacher()
ck = torch.load(os.path.join(OUT_DIR, "teacher_best.pth"), map_location=device)
teacher.load_state_dict(ck["state_dict"]); teacher.eval()
for p in teacher.parameters(): p.requires_grad_(False)
print(f"Teacher: best val Dice {ck['val_dice']:.4f} at epoch {ck['epoch']}")



In [ ]:
# %% [10] Training KD student
seed_everything()
student = build_unet()
print("student params:", n_params(student))                         # expect 5,646,402
train(student, "student_kd", epochs=80, lr=1e-4, wd=1e-5, teacher=teacher)



In [ ]:
# %% [11] Training baseline
seed_everything()
baseline = build_unet()
train(baseline, "baseline", epochs=80, lr=1e-4, wd=1e-5)



In [ ]:
# %% [12a] ablation student, lambda_KD = 0.3
seed_everything()
student_l03 = build_unet()
train(student_l03, "student_kd_l03", epochs=80, lr=1e-4, wd=1e-5, teacher=teacher,
      lam_sup=0.70, lam_kd=0.30)


In [ ]:
# %% [12b] ablation student, lambda_KD = 0.5
seed_everything()
student_l05 = build_unet()
train(student_l05, "student_kd_l05", epochs=80, lr=1e-4, wd=1e-5, teacher=teacher,
      lam_sup=0.50, lam_kd=0.50)


In [ ]:
# %% [KL-1] Training the KL-distilled student
KD_T, LAM_SUP_KL, LAM_KD_KL = 4.0, 0.5, 0.5

def train_kl(model, name, teacher, epochs=80, lr=1e-4, wd=1e-5,
             lam_sup=LAM_SUP_KL, lam_kd=LAM_KD_KL, T=KD_T, use_amp=True, clip=1.0):
    ckpt_best = os.path.join(OUT_DIR, f"{name}_best.pth")
    ckpt_last = os.path.join(OUT_DIR, f"{name}_last.pth")
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    start, best, hist = 0, -1.0, []

    if os.path.exists(ckpt_last):
        s = torch.load(ckpt_last, map_location=device)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"])
        sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
        start, best, hist = s["epoch"] + 1, s["best"], s["hist"]
        print(f"Resuming {name} from epoch {start + 1}")

    for ep in range(start, epochs):
        model.train(); tl, t0 = 0.0, time.time()
        for img, seg, _ in train_loader:
            img, seg = img.to(device), seg.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=use_amp):
                logits = model(img)
                loss = lam_sup * sup_loss(logits, seg)
                with torch.no_grad():
                    t_logits = teacher(img)
            # per-voxel KL(teacher || student), temperature-scaled, averaged over voxels
            s_log = F.log_softmax(logits.float() / T, 1)
            t_p   = F.softmax(t_logits.float() / T, 1)
            loss_kd = F.kl_div(s_log, t_p, reduction="none").sum(1).mean() * T ** 2
            loss = loss + lam_kd * loss_kd

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
            tl += loss.item()
        sched.step()

        model.eval()
        with torch.no_grad():
            vd = float(np.mean([fg_dice(model(i.to(device)), s.to(device)) for i, s, _ in val_loader]))
        hist.append(dict(epoch=ep + 1, train_loss=tl / len(train_loader), val_dice=vd))
        print(f"{name} | epoch {ep + 1}/{epochs} | loss {tl / len(train_loader):.4f} | "
              f"val Dice {vd:.4f} | best {max(best, vd):.4f} | {time.time() - t0:.0f}s")

        if vd > best:
            best = vd
            torch.save({"state_dict": model.state_dict(), "epoch": ep + 1, "val_dice": vd}, ckpt_best)
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep, best=best, hist=hist), ckpt_last)

    pd.DataFrame(hist).to_csv(os.path.join(OUT_DIR, f"{name}_history.csv"), index=False)
    print(f"{name}: best val Dice {best:.4f}")
    return best

assert OUT_DIR.endswith("Models_v6"), OUT_DIR
assert not any(p.requires_grad for p in teacher.parameters()), "Run cell 9 first (frozen teacher)"
seed_everything()
student_kl = build_unet()
train_kl(student_kl, "student_kd_kl", teacher)



In [ ]:

# %% [KL-2] Test evaluation of the KL student + comparison with the existing results

from monai.transforms import KeepLargestConnectedComponent
from monai.metrics import compute_hausdorff_distance
klcc1 = KeepLargestConnectedComponent(applied_labels=[1], num_components=1)

@torch.no_grad()
def evaluate_kl(model, loader, n_warmup=3):
    img0 = next(iter(loader))[0].to(device)
    for _ in range(n_warmup): model(img0)
    rows = []
    for img, seg, sid in loader:
        img, seg = img.to(device), seg.to(device)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        logits = model(img)
        torch.cuda.synchronize(); t = time.perf_counter() - t0
        gt       = seg[0, 1] > 0.5
        pred_raw = logits.argmax(1)[0] == 1
        pred     = torch.as_tensor(klcc1(logits.argmax(1, keepdim=True)[0])).to(device)[0] == 1
        tp = (pred & gt).sum().item();  fp = (pred & ~gt).sum().item()
        fn = (~pred & gt).sum().item(); tn = (~pred & ~gt).sum().item()
        dice_raw = 2 * (pred_raw & gt).sum().item() / (pred_raw.sum().item() + gt.sum().item())
        pred_oh = F.one_hot(pred.long(), 2).permute(3, 0, 1, 2).unsqueeze(0).float()
        hd95 = compute_hausdorff_distance(pred_oh, seg, include_background=False, percentile=95).item()
        rows.append(dict(subject=sid[0], dice=2 * tp / (2 * tp + fp + fn), dice_raw=dice_raw,
                         sensitivity=tp / (tp + fn), precision=tp / (tp + fp) if (tp + fp) else 0.0,
                         specificity=tn / (tn + fp), accuracy=(tp + tn) / (tp + tn + fp + fn),
                         hd95_mm=hd95, vol_pred_ml=(tp + fp) / 1000, vol_gt_ml=(tp + fn) / 1000, time_s=t))
    return pd.DataFrame(rows)

m_kl = build_unet()
ck = torch.load(os.path.join(OUT_DIR, "student_kd_kl_best.pth"), map_location=device)
m_kl.load_state_dict(ck["state_dict"]); m_kl.eval()
df_kl = evaluate_kl(m_kl, test_loader)
df_kl["cat"] = df_kl.subject.map({p["subject"]: p["cat"] for p in split["test"]})
df_kl.to_csv(os.path.join(OUT_DIR, "test_per_subject_student_kd_kl.csv"), index=False)

q1, q3 = df_kl.dice.quantile([0.25, 0.75])
print(f"KD 3D U-Net (KL, T={KD_T:g}, λ={LAM_KD_KL:g}) | best val Dice {ck['val_dice']:.4f} (epoch {ck['epoch']})")
for m in ["dice", "dice_raw", "sensitivity", "precision", "hd95_mm", "time_s"]:
    print(f"  {m:12s} {df_kl[m].mean():.4f} ± {df_kl[m].std():.4f}")
print(f"  dice median [IQR] {df_kl.dice.median():.4f} [{q1:.4f}–{q3:.4f}]")
print(f"  n_fail(<0.5)      {int((df_kl.dice < 0.5).sum())}")
print(f"  volume_r          {pearsonr(df_kl.vol_pred_ml, df_kl.vol_gt_ml)[0]:.4f}")
print(f"  abs_vol_err_%     {(100 * (df_kl.vol_pred_ml - df_kl.vol_gt_ml).abs() / df_kl.vol_gt_ml).mean():.2f}")
print("\nPer-diagnosis mean Dice:", df_kl.groupby("cat").dice.mean().round(4).to_dict())

print("\nPaired Wilcoxon tests (per-subject Dice, matched by subject):")
for other in ["baseline", "student_kd", "student_kd_l03", "student_kd_l05", "teacher"]:
    f = os.path.join(OUT_DIR, f"test_per_subject_{other}.csv")
    if not os.path.exists(f):
        continue
    o = pd.read_csv(f)[["subject", "dice"]].merge(df_kl[["subject", "dice"]], on="subject", suffixes=("_o", "_kl"))
    assert len(o) == len(df_kl), f"subject mismatch with {other}"
    d = o.dice_kl - o.dice_o
    print(f"  KL student vs {other:15s}: mean diff {d.mean():+.4f}, p = {wilcoxon(o.dice_kl, o.dice_o).pvalue:.4g}")

In [ ]:
# %% [13] Test-set evaluation
klcc = KeepLargestConnectedComponent(applied_labels=[1], num_components=1)   # one hippocampus per volume

def load_best(model, name):
    model.load_state_dict(torch.load(os.path.join(OUT_DIR, f"{name}_best.pth"), map_location=device)["state_dict"])
    return model.eval()

@torch.no_grad()
def evaluate(model, loader, n_warmup=3):
    img0 = next(iter(loader))[0].to(device)
    for _ in range(n_warmup): model(img0)                      # warm-up, excluded from timing
    rows = []
    for img, seg, sid in loader:
        img, seg = img.to(device), seg.to(device)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        logits = model(img)                                     # FP32 inference
        torch.cuda.synchronize(); t = time.perf_counter() - t0

        gt       = seg[0, 1] > 0.5
        pred_raw = logits.argmax(1)[0] == 1
        pred     = torch.as_tensor(klcc(logits.argmax(1, keepdim=True)[0])).to(device)[0] == 1

        tp = (pred & gt).sum().item();  fp = (pred & ~gt).sum().item()
        fn = (~pred & gt).sum().item(); tn = (~pred & ~gt).sum().item()
        dice_raw = 2 * (pred_raw & gt).sum().item() / (pred_raw.sum().item() + gt.sum().item())

        pred_oh = F.one_hot(pred.long(), 2).permute(3, 0, 1, 2).unsqueeze(0).float()
        hd95 = compute_hausdorff_distance(pred_oh, seg, include_background=False, percentile=95).item()
        rows.append(dict(subject=sid[0], dice=2 * tp / (2 * tp + fp + fn), dice_raw=dice_raw,
                         sensitivity=tp / (tp + fn), precision=tp / (tp + fp) if (tp + fp) else 0.0,
                         specificity=tn / (tn + fp), accuracy=(tp + tn) / (tp + tn + fp + fn),
                         hd95_mm=hd95, vol_pred_ml=(tp + fp) / 1000, vol_gt_ml=(tp + fn) / 1000, time_s=t))
    return pd.DataFrame(rows)

LABELS = {"teacher": "SwinUNETR (Teacher)", "student_kd": "KD 3D U-Net (λ=0.1)",
          "student_kd_l03": "KD 3D U-Net (λ=0.3)", "student_kd_l05": "KD 3D U-Net (λ=0.5)",
          "baseline": "3D U-Net (Baseline, λ=0)"}
builders = {"teacher": build_teacher}
models = {}
for k in LABELS:
    if os.path.exists(os.path.join(OUT_DIR, f"{k}_best.pth")):
        models[k] = load_best(builders.get(k, build_unet)(), k)
print("Evaluating:", list(models))
res = {k: evaluate(m, test_loader) for k, m in models.items()}
cat_of = {p["subject"]: p["cat"] for p in split["test"]}
for k, df in res.items():
    df["cat"] = df.subject.map(cat_of)
    df.to_csv(os.path.join(OUT_DIR, f"test_per_subject_{k}.csv"), index=False)

In [ ]:
# %% [14] Summary, statistics, sanity checks
pd.set_option("display.width", 250)
summary = []
for k, df in res.items():
    r = {"Model": LABELS[k]}
    for m in ["dice", "dice_raw", "sensitivity", "precision", "hd95_mm", "time_s"]:
        r[m] = f"{df[m].mean():.4f} ± {df[m].std():.4f}"
    q1, q3 = df.dice.quantile([0.25, 0.75])
    r["dice_median[IQR]"] = f"{df.dice.median():.4f} [{q1:.4f}–{q3:.4f}]"
    r["n_fail(<0.5)"]     = int((df.dice < 0.5).sum())
    r["volume_r"]         = round(pearsonr(df.vol_pred_ml, df.vol_gt_ml)[0], 4)
    r["abs_vol_err_%"]    = round((100 * (df.vol_pred_ml - df.vol_gt_ml).abs() / df.vol_gt_ml).mean(), 2)
    r["params"]           = n_params(models[k])
    summary.append(r)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, "test_summary.csv"), index=False)
print(summary.to_string(index=False))

print("\nPer-diagnosis mean Dice:")
print(pd.DataFrame({LABELS[k]: df.groupby("cat").dice.mean() for k, df in res.items()}).round(4).to_string())

print("\nPaired Wilcoxon tests on per-subject Dice:")
pairs = [("student_kd", "baseline"), ("teacher", "student_kd"), ("teacher", "baseline"),
         ("student_kd_l03", "baseline"), ("student_kd_l05", "baseline")]
for a, b in pairs:
    if a in res and b in res:
        d = res[a].dice.values - res[b].dice.values
        print(f"  {LABELS[a]} vs {LABELS[b]}: mean diff {d.mean():+.4f}, "
              f"p = {wilcoxon(res[a].dice.values, res[b].dice.values).pvalue:.4g}")

print("\nParameter reduction: %.2fx" % (n_params(models["teacher"]) / n_params(models["student_kd"])))
print("Inference speed-up:  %.2fx" % (res["teacher"].time_s.mean() / res["student_kd"].time_s.mean()))
for k in models:
    print(f"{k}: best val Dice {torch.load(os.path.join(OUT_DIR, f'{k}_best.pth'))['val_dice']:.4f}")

for k, df in res.items():
    assert df.dice.mean() < df.accuracy.mean() - 0.01, f"{k}: Dice ≈ accuracy — metric is wrong"
print("Sanity checks passed.")



In [ ]:
seed_everything()
student_kl01 = build_unet()
train_kl(student_kl01, "student_kd_kl_l01", teacher, lam_sup=0.9, lam_kd=0.1)

In [ ]:
def report(name, label):
    m = build_unet(); ck = torch.load(os.path.join(OUT_DIR, f"{name}_best.pth"), map_location=device)
    m.load_state_dict(ck["state_dict"]); m.eval()
    df = evaluate_kl(m, test_loader); df["cat"] = df.subject.map({p["subject"]: p["cat"] for p in split["test"]})
    df.to_csv(os.path.join(OUT_DIR, f"test_per_subject_{name}.csv"), index=False)
    print(f"{label} | best val {ck['val_dice']:.4f} (epoch {ck['epoch']}) | test Dice {df.dice.mean():.4f} ± {df.dice.std():.4f}"
          f" | median {df.dice.median():.4f} | HD95 {df.hd95_mm.mean():.2f} | fails {int((df.dice < 0.5).sum())}")
    for other in ["baseline", "student_kd", "student_kd_kl", "teacher"]:
        f = os.path.join(OUT_DIR, f"test_per_subject_{other}.csv")
        if os.path.exists(f):
            o = pd.read_csv(f)[["subject", "dice"]].merge(df[["subject", "dice"]], on="subject", suffixes=("_o", "_n"))
            print(f"  vs {other:14s}: diff {(o.dice_n - o.dice_o).mean():+.4f}, p = {wilcoxon(o.dice_n, o.dice_o).pvalue:.4g}")
report("student_kd_kl_l01", "KD 3D U-Net (KL, T=4, λ=0.1)")

In [ ]:

# %% [A0] Setup for the analyses
import os, numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from monai.transforms import KeepLargestConnectedComponent

assert OUT_DIR.endswith("Models_v6"), OUT_DIR
FIG_DIR = os.path.join(OUT_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)

NAMES = {"teacher": "SwinUNETR (Teacher)",
         "baseline": "U-Net baseline (λ=0)",
         "student_kd": "KD-MSE λ=0.1",
         "student_kd_l03": "KD-MSE λ=0.3",
         "student_kd_l05": "KD-MSE λ=0.5",
         "student_kd_kl_l01": "KD-KL λ=0.1",
         "student_kd_kl": "KD-KL λ=0.5"}
NAMES = {k: v for k, v in NAMES.items() if os.path.exists(os.path.join(OUT_DIR, f"test_per_subject_{k}.csv"))}
res = {k: pd.read_csv(os.path.join(OUT_DIR, f"test_per_subject_{k}.csv")) for k in NAMES}
print("Models found:", list(NAMES))



In [ ]:
# %% [A1] All distillation variants vs baseline: paired Wilcoxon + Holm correction
base = res["baseline"][["subject", "dice"]]
rows = []
for k in NAMES:
    if k in ("baseline", "teacher"):
        continue
    m = res[k][["subject", "dice"]].merge(base, on="subject", suffixes=("", "_b"))
    rows.append(dict(model=NAMES[k], test_dice=m.dice.mean(), diff=(m.dice - m.dice_b).mean(),
                     p=wilcoxon(m.dice, m.dice_b).pvalue))
tab = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
running, adj = 0.0, []
for i, p in enumerate(tab.p):                                   # Holm step-down
    running = max(running, min(1.0, (len(tab) - i) * p)); adj.append(running)
tab["p_holm"] = adj
tab = tab.sort_values("model")
print("Distillation variants vs identically trained baseline (test set):")
print(tab.round({"test_dice": 4, "diff": 4}).to_string(index=False, formatters={"p": "{:.3g}".format, "p_holm": "{:.3g}".format}))
tab.to_csv(os.path.join(OUT_DIR, "kd_variants_vs_baseline.csv"), index=False)



In [ ]:
# %% [A2] Subjects with Dice < 0.5 in any model
wide = pd.DataFrame({NAMES[k]: res[k].set_index("subject").dice for k in NAMES})
wide["cat"] = res["baseline"].set_index("subject").cat
print("Subjects where any model has Dice < 0.5:")
print(wide[wide.drop(columns="cat").min(axis=1) < 0.5].round(3).to_string())



In [ ]:
# %% [A3] Inference pass: student–teacher agreement + effect of NOT handling the annotated side
klcc1 = KeepLargestConnectedComponent(applied_labels=[1], num_components=1)
pp = lambda logits: torch.as_tensor(klcc1(logits.argmax(1, keepdim=True)[0])).to(device)[0] == 1
def dice(a, b):
    s = a.sum().item() + b.sum().item()
    return 2 * (a & b).sum().item() / s if s else 1.0

def load(k):
    m = build_teacher() if k == "teacher" else build_unet()
    m.load_state_dict(torch.load(os.path.join(OUT_DIR, f"{k}_best.pth"), map_location=device)["state_dict"])
    return m.eval()
models = {k: load(k) for k in NAMES}

class HarPNoMirror(HarPDataset):
    """Same preprocessing as the main pipeline, but WITHOUT mirroring to the canonical side."""
    def __getitem__(self, i):
        p = self.pairs[i]
        img  = self._crop(self._load(p["brain"])).clone()
        mask = img > 0
        img[mask] = (img[mask] - img[mask].mean()) / (img[mask].std() + 1e-8)
        lbl = self._crop(self._load(p["seg"])).long()
        seg = F.one_hot(lbl, 2).permute(3, 0, 1, 2).float()
        return img.unsqueeze(0), seg, p["subject"]
nomirror_loader = torch.utils.data.DataLoader(HarPNoMirror(split["test"]), batch_size=1, shuffle=False)

agree_rows, nm_rows = [], []
with torch.no_grad():
    for (img, seg, sid), (img_n, seg_n, sid_n) in zip(test_loader, nomirror_loader):
        assert sid[0] == sid_n[0]
        img, gt = img.to(device), seg[0, 1].to(device) > 0.5
        preds = {k: pp(m(img)) for k, m in models.items()}
        for k in NAMES:
            if k != "teacher":
                agree_rows.append(dict(subject=sid[0], model=k, agree_teacher=dice(preds[k], preds["teacher"]),
                                       dice_gt=dice(preds[k], gt)))
        img_n, gt_n = img_n.to(device), seg_n[0, 1].to(device) > 0.5
        for k in ["teacher", "student_kd", "baseline"]:
            nm_rows.append(dict(subject=sid[0], model=k, dice_no_mirror=dice(pp(models[k](img_n)), gt_n)))

agree = pd.DataFrame(agree_rows)
print("Student–teacher agreement (Dice between U-Net and teacher predictions, test set):")
print(agree.groupby("model").agg(agree_mean=("agree_teacher", "mean"), agree_sd=("agree_teacher", "std"),
                                 dice_gt=("dice_gt", "mean"))
           .rename(index=NAMES).round(4).to_string())
agree.to_csv(os.path.join(OUT_DIR, "student_teacher_agreement.csv"), index=False)

nm = pd.DataFrame(nm_rows)
print("\nEffect of ignoring the annotated hemisphere (no canonical mirroring at test time):")
for k in ["teacher", "student_kd", "baseline"]:
    d_nm = nm[nm.model == k].dice_no_mirror
    d_c  = res[k].dice
    print(f"  {NAMES[k]:24s} canonical {d_c.mean():.4f} ± {d_c.std():.4f} (fails {int((d_c < 0.5).sum())})"
          f"  |  unmirrored {d_nm.mean():.4f} ± {d_nm.std():.4f} (fails {int((d_nm < 0.5).sum())})")
nm.to_csv(os.path.join(OUT_DIR, "no_mirror_eval.csv"), index=False)



In [ ]:
# %% [A4] Volume agreement: Bland–Altman (teacher, MSE λ=0.1 student, baseline)
show = [k for k in ["teacher", "student_kd", "baseline"] if k in res]
fig, ax = plt.subplots(1, len(show), figsize=(5 * len(show), 4.2), sharey=True)
print("Bland–Altman (predicted − manual volume, mL):")
for a, k in zip(np.atleast_1d(ax), show):
    df = res[k]; mean_v = (df.vol_pred_ml + df.vol_gt_ml) / 2; diff_v = df.vol_pred_ml - df.vol_gt_ml
    bias, sd = diff_v.mean(), diff_v.std(); lo, hi = bias - 1.96 * sd, bias + 1.96 * sd
    a.scatter(mean_v, diff_v, s=18)
    for y, ls in [(bias, "-"), (lo, "--"), (hi, "--")]: a.axhline(y, ls=ls, color="k", lw=1)
    a.set_title(NAMES[k]); a.set_xlabel("Mean of predicted and manual volume (mL)")
    print(f"  {NAMES[k]:24s} bias {bias:+.3f} mL | limits of agreement [{lo:+.3f}, {hi:+.3f}] mL")
np.atleast_1d(ax)[0].set_ylabel("Predicted − manual (mL)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "bland_altman.png"), dpi=200); plt.show()



In [ ]:
# %% [A5] Convergence: validation curves, onset of foreground prediction, epochs to reach 0.80
def history(k):
    f = os.path.join(OUT_DIR, f"{k}_history.csv")
    if os.path.exists(f): return pd.read_csv(f)
    return pd.DataFrame(torch.load(os.path.join(OUT_DIR, f"{k}_last.pth"), map_location="cpu")["hist"])

conv = []
fig, ax = plt.subplots(figsize=(8, 5))
for k in NAMES:
    h = history(k)
    onset = h.loc[h.val_dice > 0.05, "epoch"].min()
    reach = h.loc[h.val_dice >= 0.80, "epoch"].min()
    conv.append(dict(model=NAMES[k], epochs_trained=len(h), first_epoch_dice_gt_0_05=onset,
                     first_epoch_dice_ge_0_80=reach, best_val=h.val_dice.max(),
                     best_epoch=int(h.loc[h.val_dice.idxmax(), "epoch"])))
    ax.plot(h.epoch, h.val_dice, label=NAMES[k], lw=1.5 if k in ("teacher", "baseline", "student_kd") else 1)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Dice"); ax.set_ylim(0, 0.95); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "validation_curves.png"), dpi=200); plt.show()
print(pd.DataFrame(conv).round(4).to_string(index=False))
pd.DataFrame(conv).to_csv(os.path.join(OUT_DIR, "convergence.csv"), index=False)
print("\nFigures saved to", FIG_DIR)

In [ ]:

# %% [X0] Setup for GradCam
import os, numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt

assert OUT_DIR.endswith("Models_v6"), OUT_DIR
FIG_DIR = os.path.join(OUT_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)

XAI_MODELS = {                                   # key: (label, target layer)
    "teacher":    ("SwinUNETR (Teacher)",  "decoder2"),
    "student_kd": ("KD-MSE λ=0.1 student", "dec2"),
    "baseline":   ("U-Net baseline",       "dec2"),
}

def load_model(k):
    m = build_teacher() if k == "teacher" else build_unet()
    m.load_state_dict(torch.load(os.path.join(OUT_DIR, f"{k}_best.pth"), map_location=device)["state_dict"])
    return m.eval()

class CAM3D:
    """Grad-CAM and Grad-CAM++ for a 3D segmentation network (one forward/backward pass gives both)."""
    def __init__(self, model, layer_name):
        self.model = model
        layer = dict(model.named_modules())[layer_name]
        self.acts, self.grads = None, None
        self.handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def remove(self):
        self.handle.remove()

    def __call__(self, img):
        img = img.clone().to(device).requires_grad_(True)       # ensures a graph even if weights are frozen
        self.model.zero_grad(set_to_none=True)
        logits = self.model(img)
        pred = logits.argmax(1)[0] == 1
        score = logits[0, 1][pred].sum() if pred.any() else logits[0, 1].sum()
        score.backward()

        A, G = self.acts.detach()[0].float(), self.grads.detach()[0].float()   # (K, d, h, w)
        # Grad-CAM: channel weight = spatial mean of the gradient
        w = G.mean(dim=(1, 2, 3))
        cam = F.relu((w[:, None, None, None] * A).sum(0))
        # Grad-CAM++ (Chattopadhay et al., 2018): voxel-wise weighting of positive gradients
        g2, g3 = G ** 2, G ** 3
        sumA  = A.sum(dim=(1, 2, 3), keepdim=True)
        alpha = g2 / (2 * g2 + sumA * g3 + 1e-7)
        alpha = torch.where(G != 0, alpha, torch.zeros_like(alpha))
        w_pp  = (alpha * F.relu(G)).sum(dim=(1, 2, 3))
        campp = F.relu((w_pp[:, None, None, None] * A).sum(0))

        maps = []
        for c in (cam, campp):
            c = F.interpolate(c[None, None], size=img.shape[2:], mode="trilinear", align_corners=False)[0, 0]
            c = (c - c.min()) / (c.max() - c.min() + 1e-8)
            maps.append(c.cpu())
        return maps[0], maps[1], pred.cpu()

xai = {k: CAM3D(load_model(k), layer) for k, (_, layer) in XAI_MODELS.items()}
test_ds = HarPDataset(split["test"])
idx_of = {p["subject"]: i for i, p in enumerate(split["test"])}
print("Ready:", list(xai))



In [ ]:
# %% [X1] Figure for one subject
#   Default: the test subject with the median baseline Dice (a representative case).
#   For the hard case, set SUBJECT = "ADNI_136_S_0429".
base_csv = pd.read_csv(os.path.join(OUT_DIR, "test_per_subject_baseline.csv"))
SUBJECT = base_csv.sort_values("dice").iloc[len(base_csv) // 2].subject

img, seg, _ = test_ds[idx_of[SUBJECT]]
gt = seg[1] > 0.5
cx, cy, cz = [int(v) for v in torch.nonzero(gt).float().mean(0).round()]
cut = lambda v, plane: {"axial": v[:, :, cz], "coronal": v[:, cy, :], "sagittal": v[cx, :, :]}[plane].T

planes = ["axial", "coronal", "sagittal"]
fig, ax = plt.subplots(len(xai), 6, figsize=(21, 3.6 * len(xai)))
for r, (k, cam_fn) in enumerate(xai.items()):
    cam, campp, pred = cam_fn(img[None])
    for p, plane in enumerate(planes):
        for j, (m, name) in enumerate([(cam, "Grad-CAM"), (campp, "Grad-CAM++")]):
            a = ax[r, 2 * p + j]
            a.imshow(cut(img[0], plane), cmap="gray", origin="lower")
            a.imshow(cut(m, plane), cmap="jet", alpha=0.45, origin="lower", vmin=0, vmax=1)
            a.contour(cut(gt.float(), plane), levels=[0.5], colors="r", linewidths=1)
            a.contour(cut(pred.float(), plane), levels=[0.5], colors="c", linewidths=1)
            a.set_xticks([]); a.set_yticks([])
            if r == 0: a.set_title(f"{name} — {plane}", fontsize=10)
    ax[r, 0].set_ylabel(XAI_MODELS[k][0], fontsize=10)
fig.suptitle(f"{SUBJECT}  (red: manual label, cyan: prediction)", fontsize=12)
plt.tight_layout()
out_png = os.path.join(FIG_DIR, f"gradcam_{SUBJECT}.png")
plt.savefig(out_png, dpi=200); plt.show()
print("Saved", out_png)



In [ ]:
# %% [X2] Quantitative localization over the full test set
#   Share of attribution mass inside the hippocampus dilated by R voxels.
#   "chance" = share expected if attribution were spread uniformly over the ROI.
R = 3
def share_inside(m, gt_mask):
    region = F.max_pool3d(gt_mask[None, None].float(), 2 * R + 1, stride=1, padding=R)[0, 0] > 0
    return (m[region].sum() / (m.sum() + 1e-8)).item(), region.float().mean().item()

rows = []
for i, p in enumerate(split["test"]):
    img, seg, sid = test_ds[i]
    gt = seg[1] > 0.5
    for k, cam_fn in xai.items():
        cam, campp, _ = cam_fn(img[None])
        s_cam, chance = share_inside(cam, gt)
        s_pp, _       = share_inside(campp, gt)
        rows.append(dict(subject=sid, model=XAI_MODELS[k][0], gradcam_inside=s_cam,
                         gradcampp_inside=s_pp, chance=chance))
loc = pd.DataFrame(rows)
loc.to_csv(os.path.join(OUT_DIR, "xai_localization.csv"), index=False)
print(f"Share of attribution inside the hippocampus (dilated {R} voxels), test set, mean ± SD:")
print(loc.groupby("model")[["gradcam_inside", "gradcampp_inside", "chance"]]
         .agg(lambda s: f"{s.mean():.3f} ± {s.std():.3f}").to_string())


In [ ]:
# %% [X3 + X4] Peak localization and cross-model similarity
from scipy.stats import spearmanr, wilcoxon

R, TOP = 3, 0.01
def region_of(gt_mask):
    return F.max_pool3d(gt_mask[None, None].float(), 2 * R + 1, stride=1, padding=R)[0, 0] > 0

def peak_metrics(m, region):
    flat_m, flat_r = m.flatten(), region.flatten()
    peak_inside = bool(flat_r[flat_m.argmax()])
    k = max(1, int(TOP * flat_m.numel()))
    top_inside = flat_r[flat_m.topk(k).indices].float().mean().item()
    return peak_inside, top_inside

ds = lambda m: F.avg_pool3d(m[None, None], 4)[0, 0].flatten().numpy()   # 4x4x4 pooling for speed
METHODS = ["Grad-CAM", "Grad-CAM++"]

peak_rows, sim_rows = [], []
for i, p in enumerate(split["test"]):
    img, seg, sid = test_ds[i]
    region = region_of(seg[1] > 0.5)
    maps = {}
    for k, cam_fn in xai.items():
        cam, campp, _ = cam_fn(img[None])
        maps[k] = (cam, campp)
        for name, m in zip(METHODS, maps[k]):
            pk, tp = peak_metrics(m, region)
            peak_rows.append(dict(subject=sid, model=XAI_MODELS[k][0], method=name,
                                  peak_inside=pk, top1pct_inside=tp))
    for j, name in enumerate(METHODS):
        t = ds(maps["teacher"][j])
        sim_rows.append(dict(subject=sid, method=name,
                             student_vs_teacher=spearmanr(ds(maps["student_kd"][j]), t)[0],
                             baseline_vs_teacher=spearmanr(ds(maps["baseline"][j]), t)[0],
                             student_vs_baseline=spearmanr(ds(maps["student_kd"][j]), ds(maps["baseline"][j]))[0]))

peak = pd.DataFrame(peak_rows); peak.to_csv(os.path.join(OUT_DIR, "xai_peak_localization.csv"), index=False)
chance = region.float().mean().item()
print(f"[X3] Peak-based localization (hippocampus dilated {R} voxels; chance ≈ {chance:.3f} of ROI):")
print(peak.groupby(["model", "method"]).agg(pointing_game=("peak_inside", "mean"),
                                            top1pct_inside=("top1pct_inside", "mean")).round(3).to_string())

sim = pd.DataFrame(sim_rows); sim.to_csv(os.path.join(OUT_DIR, "xai_map_similarity.csv"), index=False)
print("\n[X4] Spearman correlation between attribution maps (mean ± SD over test subjects):")
for name in METHODS:
    s = sim[sim.method == name]
    p = wilcoxon(s.student_vs_teacher, s.baseline_vs_teacher).pvalue
    print(f"  {name:10s} student–teacher {s.student_vs_teacher.mean():.3f} ± {s.student_vs_teacher.std():.3f} | "
          f"baseline–teacher {s.baseline_vs_teacher.mean():.3f} ± {s.baseline_vs_teacher.std():.3f} | "
          f"student–baseline {s.student_vs_baseline.mean():.3f} ± {s.student_vs_baseline.std():.3f} | "
          f"student vs baseline similarity to teacher: p = {p:.3g}")



In [ ]:
# %% [X5] Randomization sanity check: trained baseline vs untrained U-Net (same architecture, same layer)
seed_everything()
rand_cam = CAM3D(build_unet().eval(), "dec2")
san_rows = []
for i, p in enumerate(split["test"]):
    img, seg, sid = test_ds[i]
    region = region_of(seg[1] > 0.5)
    tr_cam, tr_pp, _ = xai["baseline"](img[None])
    rd_cam, rd_pp, _ = rand_cam(img[None])
    for name, tr, rd in [("Grad-CAM", tr_cam, rd_cam), ("Grad-CAM++", tr_pp, rd_pp)]:
        san_rows.append(dict(subject=sid, method=name,
                             spearman_trained_vs_random=spearmanr(ds(tr), ds(rd))[0],
                             inside_trained=(tr[region].sum() / (tr.sum() + 1e-8)).item(),
                             inside_random=(rd[region].sum() / (rd.sum() + 1e-8)).item()))
rand_cam.remove()
san = pd.DataFrame(san_rows); san.to_csv(os.path.join(OUT_DIR, "xai_randomization_check.csv"), index=False)
print(f"[X5] Randomization sanity check (chance inside-share ≈ {chance:.3f}):")
print(san.groupby("method")[["spearman_trained_vs_random", "inside_trained", "inside_random"]]
         .agg(lambda s: f"{s.mean():.3f} ± {s.std():.3f}").to_string())

for c in xai.values(): c.remove()

In [ ]:
drive.flush_and_unmount()